In [0]:
%pip install azure-storage-file-datalake azure-identity pandas python-dotenv
dbutils.library.restartPython()

In [0]:
# Inicialização da conexão
import os
import sys
from dotenv import load_dotenv

# FORÇAR O DATABRICKS A OLHAR O DIRETÓRIO DO NOTEBOOK ATUAL
notebook_path = os.path.dirname(os.path.abspath(sys.argv[0]))
env_path = os.path.join(notebook_path, ".env")

# Carrega o arquivo apontando para o caminho exato
load_dotenv(dotenv_path=env_path)

# Captura e validação das variáveis
CLIENT_ID       = os.getenv("CLIENT_ID")
CLIENT_SECRET   = os.getenv("CLIENT_SECRET")
TENANT_ID       = os.getenv("TENANT_ID")
STORAGE_ACCOUNT = os.getenv("STORAGE_ACCOUNT")
CONTAINER       = os.getenv("CONTAINER")

if not CLIENT_ID:
    load_dotenv()
    CLIENT_ID       = os.getenv("CLIENT_ID")
    CLIENT_SECRET   = os.getenv("CLIENT_SECRET")
    TENANT_ID       = os.getenv("TENANT_ID")
    STORAGE_ACCOUNT = os.getenv("STORAGE_ACCOUNT")
    CONTAINER       = os.getenv("CONTAINER")

# Validação crítica para evitar o ValueError
if not all([CLIENT_ID, CLIENT_SECRET, TENANT_ID, STORAGE_ACCOUNT, CONTAINER]):
    raise ValueError(
        f"❌ ERRO: O arquivo .env não foi localizado ou está incompleto!\n"
        f"Caminho tentado: {env_path}\n"
    )

print("✅ Variáveis carregadas com sucesso via arquivo .env!")
print(f"   Configurando Spark para o Storage: {STORAGE_ACCOUNT} -> Container: {CONTAINER}\n")


# 🔄 FUNÇÃO UTILIZANDO SPARK (URL TOTALMENTE CORRIGIDA)
def ler_csv_adls(caminho_arquivo):
    """
    Lê um arquivo CSV diretamente do Azure Data Lake Gen2 usando Spark DataFrame,
    passando as credenciais de autenticação baseadas no Service Principal em tempo de execução.
    """
    # 1. Monta a URL do Data Lake
    url_adls = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/{caminho_arquivo}"
    
    # 2. Monta o endpoint da Microsoft garantindo a URL perfeitamente limpa
    endpoint_microsoft = f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/token"
    
    # 3. Executa a leitura no Spark passando as configurações
    df = spark.read \
        .format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .option("fs.azure.account.auth.type", "OAuth") \
        .option("fs.azure.account.oauth.provider.type", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider") \
        .option("fs.azure.account.oauth2.client.id", CLIENT_ID) \
        .option("fs.azure.account.oauth2.client.secret", CLIENT_SECRET) \
        .option("fs.azure.account.oauth2.client.endpoint", endpoint_microsoft) \
        .load(url_adls)
        
    return df

print("✨ Nova função carregada com sucesso e isolada em memória!")

In [0]:
# Recarrega o DataFrame com a função atualizada
df_pedidos = ler_csv_adls("batch-data/ecommerce_pedidos.csv")

# Exibe as primeiras linhas
display(df_pedidos)

In [0]:
# Analisa min, max, média e desvio padrão das colunas de valor
display(df_pedidos.select("valor_total", "valor_frete").summary())


In [0]:
from pyspark.sql.functions import col, count, round

# Análise de Status dos Pedidos
total_registros = df_pedidos.count()

df_pedidos.groupBy("status_pedido") \
    .agg(count("*").alias("quantidade")) \
    .withColumn("percentual", round((col("quantidade") / total_registros) * 100, 2)) \
    .orderBy(col("quantidade").desc()) \
    .show()


In [0]:
from pyspark.sql.functions import min, max

# Descobre a data do primeiro e do último pedido da base
df_pedidos.select(
    min("dt_pedido").alias("data_primeiro_pedido"),
    max("dt_pedido").alias("data_ultimo_pedido")
).show()


In [0]:
from pyspark.sql.functions import datediff, col

# Calcula a diferença em dias entre a data do pedido e a previsão de entrega
df_prazos = df_pedidos.withColumn("dias_prazo_estimado", datediff(col("dt_previsao_entrega"), col("dt_pedido")))

display(df_prazos.select("dias_prazo_estimado").summary())


In [0]:
from pyspark.sql.functions import col, sum

# Soma quantos valores nulos existem em cada coluna essencial
df_pedidos.select([sum(col(c).isNull().cast("int")).alias(c) for c in df_pedidos.columns]).show()

In [0]:
# Recarrega o DataFrame com a função atualizada
df_clientes = ler_csv_adls("batch-data/ecommerce_clientes.csv")

# Exibe as primeiras linhas
display(df_clientes)

In [0]:
from pyspark.sql.functions import col, count

print("VALIDAÇÃO DE INTEGRIDADE: PEDIDOS X CLIENTES")

# 1. Encontra o id_cliente com o maior número de pedidos
pedido_top_cliente = df_pedidos.groupBy("id_cliente") \
    .agg(count("*").alias("total_pedidos")) \
    .orderBy(col("total_pedidos").desc()) \
    .first() # Pega apenas a primeira linha (o campeão)

if pedido_top_cliente:
    top_id_cliente = pedido_top_cliente["id_cliente"]
    qtd_pedidos    = pedido_top_cliente["total_pedidos"]
    
    print(f"🥇 O cliente que mais comprou na base tem o id_cliente: {top_id_cliente}")
    print(f"📦 Ele/Ela realizou um total de {qtd_pedidos} pedidos.\n")
    
    # 2. Verifica se este ID específico existe na tabela de clientes
    cliente_existe = df_clientes.filter(col("id_cliente") == top_id_cliente).count() > 0
    
    if cliente_existe:
        # Busca o nome dele para confirmar visualmente
        dados_cliente = df_clientes.filter(col("id_cliente") == top_id_cliente).select("nome", "sobrenome", "email").first()
        print(f"✅ SUCESSO! O ID {top_id_cliente} EXISTE na tabela de clientes.")
        print(f"👤 Cliente cadastrado: {dados_cliente['nome']} {dados_cliente['sobrenome']} ({dados_cliente['email']})")
    else:
        print(f"❌ ALERTA DE INTEGRIDADE! O ID {top_id_cliente} NÃO existe na tabela de clientes. É um dado órfão.")
else:
    print("ℹ️ A tabela de pedidos está vazia.")

In [0]:
from pyspark.sql.functions import col, count, max, min, avg

print("ANÁLISE DE COMPORTAMENTO: RECORRÊNCIA DE CLIENTES")

# Agrupa todos os clientes e conta quantos pedidos cada um tem
df_recorrencia = df_pedidos.groupBy("id_cliente") \
    .agg(count("*").alias("total_pedidos"))

# Calcula as métricas de controle (máximo, mínimo e média de pedidos por cliente)
metricas = df_recorrencia.select(
    min("total_pedidos").alias("minimo"),
    max("total_pedidos").alias("maximo"),
    avg("total_pedidos").alias("media")
).first()

print(f"📊 Distribuição de pedidos por cliente:")
print(f"   • Menor número de pedidos de um cliente: {metricas['minimo']}")
print(f"   • Maior número de pedidos de um cliente: {metricas['maximo']}")
print(f"   • Média de pedidos por cliente:          {metricas['media']:.2f}\n")

if metricas['maximo'] == 1:
    print("💡 CONCLUSÃO EDA: Esta base de dados NÃO possui recorrência. Todos os 10 mil clientes compraram exatamente 1 única vez.")
else:
    print("💡 CONCLUSÃO EDA: Existe recorrência na base. O topo mostrou 1 porque empatou, mas há clientes com mais compras.")
